In [ ]:
import xarray as xr
import torch
import numpy as np
import datetime
import cftime
import matplotlib.pyplot as plt
import sys
import re
import matplotlib.colors as colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean 
import fsspec
import xesmf as xe
from scipy.stats import pearsonr

In [ ]:
# --- Configuration ---
# File System
fs = fsspec.filesystem("s3", profile="osn", client_kwargs={"endpoint_url": "https://nyu1.osn.mghpcc.org"})
storage_options = {"profile": "osn", "client_kwargs": {"endpoint_url": "https://nyu1.osn.mghpcc.org"}}

# Paths
# TRUTH_PATH = "s3://emulators/am16581/data/2025-11/om4_halfdeg_v4/OM4.zarr" 
TRUTH_PATH = "/pscratch/sd/a/asubel/Ocean_Emulator/notebooks/temp/Sample_Rollouts/Half_Degree_Subset.zarr" 

PRED_PATH_1 = '/pscratch/sd/a/asubel/Ocean_Emulator/notebooks/temp/Sample_Rollouts/halfdeg_eval_preds.zarr'

# Set PRED_PATH_2 to None if you only want to compare one emulator
PRED_PATH_2 = '/pscratch/sd/a/asubel/Ocean_Emulator/notebooks/temp/Sample_Rollouts/halfdeg_eval_preds_v1.zarr'
# PRED_PATH_2 = None 

# Analysis & Plotting Config
EMULATOR_1_LABEL = "Emulator WDMSE"
EMULATOR_1_COLOR = "tab:orange"

EMULATOR_2_LABEL = "Emulator V1"
EMULATOR_2_COLOR = "tab:green"

TRUTH_LABEL = "(OM4)"
TRUTH_COLOR = "black"

# Auxiliary Data Paths
LEVS_PATH = '/pscratch/sd/a/asubel/Data/Chapter_2/CM4_CMIP_New.zarr/'
OCEAN_SAMPLE_PATH = '/pscratch/sd/a/asubel/full-model/scratch/ocean_sample.nc'
DEPTHO_PATH = '/pscratch/sd/a/asubel/full-model/scratch/deptho_unfiltered.nc'

# Variable mapping
OCEAN_FIELD_NAME_PREFIXES = {
    'thetao': ['thetao_'], 'so': ['so_'], 'uo': ['uo_'], 'vo': ['vo_'],
    'zos': ['zos'], 'sea_surface_temperature': ['sst'], 'lev': ['idepth_']
}

In [ ]:
def concat_levels(ds: xr.Dataset, prefix: str, xg_name: str | None = None, new_coord: None = None) -> xr.DataArray:
    """Concatenates the levels of a 3D variable into a single DataArray."""
    levels = []
    names = []
    level_pattern = re.compile(rf"{prefix}_(\d+)$")
    for name in ds.data_vars:
        match = level_pattern.search(name)
        if match:
            levels.append(int(match.group(1)))
            names.append(name)
    names = sorted(names, key=lambda name: levels[names.index(name)])
    da = xr.concat([ds[name] for name in names], dim="lev")
    da.name = prefix
    if xg_name is not None:
        da.attrs["xg_name"] = xg_name
    if new_coord is not None:
        new_coord_name, new_coord = new_coord
        da[new_coord_name] = new_coord
        da = da.swap_dims({"lev": new_coord_name})
    return da

def extract_basin_masks(ds_with_basin_var):
    """Extracts boolean masks from the integer 'basin' variable."""
    basin_code = ds_with_basin_var['basin']
    return {
        'Atlantic': xr.where(basin_code == 2, 1.0, np.nan),
        'Pacific': xr.where(basin_code == 3, 1.0, np.nan),
        'Indian': xr.where(basin_code == 5, 1.0, np.nan),
        'Southern': xr.where(basin_code == 1, 1.0, np.nan),
    }

def _detrend_linear_torch(data):
    """
    Removes a linear plane of best fit from 4D (B, C, H, W) data.
    """
    B, C, H, W = data.shape
    device = data.device
    dtype = data.dtype
    
    y_coords = torch.linspace(-1, 1, H, device=device, dtype=dtype)
    x_coords = torch.linspace(-1, 1, W, device=device, dtype=dtype)
    Y, X = torch.meshgrid(y_coords, x_coords, indexing='ij')
    
    A = torch.stack([X.flatten(), Y.flatten(), torch.ones_like(X).flatten()], dim=1)
    
    B_prime = B * C
    data_flat = data.reshape(B_prime, H * W)
    

    coeffs, _, _, _ = torch.linalg.lstsq(A, data_flat.T)

    plane = (A @ coeffs.permute(1, 0).unsqueeze(-1)).reshape(B_prime, H, W)
    
    detrended_data = data.reshape(B_prime, H, W) - plane
    
    return detrended_data.reshape(B, C, H, W)


def compute_isotropic_spectrum_torch(data, dx=1.0, dy=1.0, num_bins=None, n_factor = 4,
                                     remove_mean=True, detrend=None, window='Hann',
                                     truncate=True, cutoff_before_bins: bool = True):
    """
    Computes the isotropic 1D power spectrum from 2D (H,W), 3D (B,H,W),
    or 4D (B,C,H,W) data. Matches `xrft.isotropic_power_spectrum(scaling="density")`.

    The output spectrum is computed for each batch and channel element.

    Parameters:
    ----------
    data : torch.Tensor
        Input data tensor. Can be 2D, 3D, or 4D.
    dx : float, optional
        Grid spacing in the x-dimension.
    dy : float, optional
        Grid spacing in the y-dimension.
    num_bins : int, optional
        Number of bins. If None, defaults to min(H, W) // n_factor.
    n_factor : int, optional
        Factor to determine number of bins.
    remove_mean : bool, optional
        If True, removes spatial mean. Overridden by `detrend`.
    detrend : str, optional
        'linear' or 'constant'.
    window : str, optional
        'hann' or 'Hann'.
    truncate : bool, optional
        If True, truncates spectrum at the smallest Nyquist frequency.
    cutoff_before_bins: bool, optional
        If True, truncates the spectrum after already computing the bin locations. Matches xrft
    Returns:
    -------
    k_bins_centers : torch.Tensor
        1D tensor of bin center wavenumbers. Shape: (num_bins,)
    iso_spectrum : torch.Tensor
        1D tensor of the (k * P(k)) spectrum.
        Shape: (B, C, num_bins), (B, num_bins), or (num_bins,)
    """
    
    device = data.device
    dtype = data.dtype
    orig_dim = data.dim()
    
    if orig_dim == 2:
        data = data.reshape(1, 1, *data.shape)  
    elif orig_dim == 3:
        data = data.unsqueeze(1)                
    elif orig_dim != 4:
        raise ValueError("Input data must be 2D, 3D, or 4D (B, C, H, W)")
        
    B, C, H, W = data.shape
    B_prime = B * C 
    Lx = W * dx
    Ly = H * dy
    
    if num_bins is None:
        num_bins = min(H, W) // n_factor 

    if detrend == 'linear':
        data = _detrend_linear_torch(data)
    elif detrend == 'constant' or remove_mean:
        data = data - torch.mean(data, dim=(-2, -1), keepdim=True)

    if window and window.lower() == 'hann':
        win_y = torch.hann_window(H, device=device, dtype=dtype).unsqueeze(1)
        win_x = torch.hann_window(W, device=device, dtype=dtype).unsqueeze(0)
        win_2d = (win_y * win_x).reshape(1, 1, H, W)
        
        window_correction = torch.mean(win_2d**2).item()
        data = data * win_2d
    else:
        window_correction = 1.0

    fft_2d = torch.fft.rfft2(data, norm='forward')
    
    power_2d = torch.abs(fft_2d)**2
    power_2d = power_2d / window_correction
    
    psd_2d = power_2d * (Lx * Ly)
    
    k_x = torch.fft.rfftfreq(W, d=dx, device=device, dtype=dtype)
    k_y = torch.fft.fftfreq(H, d=dy, device=device, dtype=dtype)
    
    k_x_nyq = 1.0 / (2.0 * dx)
    k_y_nyq = 1.0 / (2.0 * dy)

    k_Y, k_X = torch.meshgrid(k_y, k_x, indexing='ij')
    k_mag = torch.sqrt(k_X**2 + k_Y**2)
    
    k_max_domain = k_mag.max()
    
    if truncate and cutoff_before_bins:
        k_max_cutoff = min(k_x_nyq, k_y_nyq)
        k_max = min(k_max_domain, k_max_cutoff)
    else:
        k_max = k_max_domain
        
    k_bins = torch.linspace(0, k_max, num_bins + 1, device=device, dtype=dtype)
    if truncate and not cutoff_before_bins:
        k_max_cutoff = min(k_x_nyq, k_y_nyq) 
        k_max = min(k_max_domain, k_max_cutoff)        
        k_bins = k_bins[k_bins<k_max_cutoff]
        num_bins = k_bins.numel()-1
    k_bins_centers = (k_bins[:-1] + k_bins[1:]) / 2

    k_mag_flat = k_mag.flatten()
    bin_edges = k_bins[1:-1] 
    
    bin_indices = torch.bucketize(k_mag_flat, bin_edges, right=True)
    
    N_flat = k_mag_flat.shape[0]
    psd_flat_batched = psd_2d.reshape(B_prime, N_flat)

    bin_indices_batched = bin_indices.expand(B_prime, -1)
    binned_psd_sum = torch.zeros(B_prime, num_bins, device=device, dtype=dtype)
    binned_psd_sum.scatter_add_(dim=1, index=bin_indices_batched, src=psd_flat_batched)
    binned_counts = torch.bincount(
        bin_indices, 
        minlength=num_bins
    )
    
    binned_counts_safe = binned_counts.float()
    binned_counts_safe[binned_counts_safe == 0] = torch.nan

    iso_psd_binned = binned_psd_sum / binned_counts_safe.unsqueeze(0)
    iso_spectrum = iso_psd_binned * k_bins_centers.unsqueeze(0)
    iso_spectrum = iso_spectrum.reshape(B, C, num_bins)
    iso_spectrum[..., 0] = torch.nan
    
    if orig_dim == 2:
        iso_spectrum = iso_spectrum.squeeze(0).squeeze(0) 
    elif orig_dim == 3:
        iso_spectrum = iso_spectrum.squeeze(1)            
    return k_bins_centers, iso_spectrum

In [ ]:
STATIC_GRID_PATH = 's3://emulators/am16581/ocean_static_no_mask_table.zarr'

# 1. Load Truth
if "s3" in TRUTH_PATH:
    data_truth_ocean = xr.open_zarr(TRUTH_PATH, storage_options=storage_options)
else:
    data_truth_ocean = xr.open_zarr(TRUTH_PATH)

# 2. Load Emulator 1
data_ocean_1 = xr.open_zarr(PRED_PATH_1).rename({'lat':'y','lon':'x'})

# 3. Load Emulator 2 (Conditional)
data_ocean_2 = None
if PRED_PATH_2 is not None:
    data_ocean_2 = xr.open_zarr(PRED_PATH_2).rename({'lat':'y','lon':'x'})

# 4. Synchronize Time Slices
time_slice_pred = slice(data_ocean_1.time.values[0], data_ocean_1.time.values[-1])
data_truth_ocean = data_truth_ocean.sel(time=time_slice_pred)

# 5. Load Aux Data (Levels, Depth)
levs = xr.open_zarr(LEVS_PATH)['lev'].values
ocean_test = xr.open_dataset(OCEAN_SAMPLE_PATH)
deptho = xr.open_dataset(DEPTHO_PATH)['deptho']

# 6. Calculate Vertical Spacing (dz)
depth_levels = concat_levels(ocean_test, 'idepth', 'levels')
dz_from_data = (depth_levels[1:] - depth_levels[:-1]).data

# 7. Grid Loading & Basin Mask Generation
# We assume grid shape matches the first emulator
shape_processed = (data_ocean_1.y.size, data_ocean_1.x.size)

try:
    print(f"Loading target grid for shape {shape_processed}...")
    target_grid_path = f"s3://emulators/am16581/grids/gaussian_grid_{shape_processed[0]}_by_{shape_processed[1]}.zarr"
    target_grid = xr.open_zarr(target_grid_path, storage_options=storage_options)
    
    # Rename coordinates to standard names
    target_grid = target_grid.rename({
        "grid_x": "x_b", "grid_y": "y_b", "grid_xt": "x", "grid_yt": "y",
        "grid_lon": "lon_b", "grid_lat": "lat_b", "grid_lont": "lon", "grid_latt": "lat",
    }).compute()
    
    # --- Basin Mask Generation (New) ---
    print("Loading and regridding basin mask from static grid...")
    ds_static = xr.open_zarr(STATIC_GRID_PATH, storage_options=storage_options)
    
    # Prepare Source (Static Grid)
    ds_source = ds_static[['basin', 'geolat', 'geolon']].rename({'geolat': 'lat', 'geolon': 'lon'})
    
    # Prepare Target (Emulator Grid)
    ds_target = target_grid[['lat', 'lon']]
    
    # Regrid (Nearest Neighbor)
    regridder = xe.Regridder(
        ds_source, 
        ds_target, 
        method='nearest_s2d', 
        periodic=True, 
        # filename='basin_weights.nc',
        reuse_weights=False
    )
    basin_regridded = regridder(ds_source['basin'])
    
    # Define Grid Metrics to be attached to all datasets
    # Note: Added 'basin' to this dictionary
    R = 6.371e6
    
    dx_deg = target_grid['lon_b'].diff('x_b')[:-1].values
    dy_deg = target_grid['lat_b'].diff('y_b')[:,:-1].values
    dx_cal = np.deg2rad(dx_deg)*np.cos(np.deg2rad(target_grid['lat'].values))*R
    dy_cal = np.deg2rad(dy_deg)*R

    grid_metrics = {
        'dx': (['y','x'], dx_cal),
        'dy': (['y','x'], dy_cal),
        'areacello': (['y','x'], xe.util.cell_area(target_grid, R).values),
        'basin': (['y','x'], basin_regridded.values) 
    }
    
except Exception as e:
    raise ValueError(f'Error loading grid or generating mask for shape {shape_processed}: {e}')

In [ ]:
def process_dataset(ds_input, truth_levs, dz_arr, grid_metrics_dict, is_truth=False):
    """Encapsulates the logic to turn raw rollout/truth into a stacked dataset with grid metrics."""
    
    # Initialize basic coordinate structure
    ds_stacked = xr.Dataset(coords={
        'time': (['time'], ds_input.time.values),
        'y': (['y'], ds_input.y.values),
        'x': (['x'], ds_input.x.values),
        'lev': (['lev'], truth_levs),
        'dz': (['lev'], dz_arr),
    })

    # Process 3D vars
    for var in ['uo', 'vo', 'thetao', 'so']:
        prefix = OCEAN_FIELD_NAME_PREFIXES[var][0][:-1]
        data_processed = concat_levels(ds_input, prefix, var).squeeze().transpose('time', 'y', 'x', 'lev').data
        ds_stacked[var] = (['time', 'y', 'x', 'lev'], data_processed)

    # Process 2D vars
    ds_stacked['zos'] = (['time', 'y', 'x'], ds_input['zos'].squeeze().data)

    # Apply Dummy Mask to dz (as done in original notebook)
    ds_stacked['dz'] = (ds_stacked['dz'] * (ds_stacked['vo'].isel(time=0) * 0 + 1)).compute()

    # Add Grid Metrics
    for metric, data in grid_metrics_dict.items():
        ds_stacked[metric] = data
        ds_stacked = ds_stacked.set_coords(metric)

    return ds_stacked

# Process all datasets
dataset_truth = process_dataset(data_truth_ocean, levs, dz_from_data, grid_metrics, is_truth=True)
dataset_emu_1 = process_dataset(data_ocean_1, levs, dz_from_data, grid_metrics)

dataset_emu_2 = None
if data_ocean_2 is not None:
    dataset_emu_2 = process_dataset(data_ocean_2, levs, dz_from_data, grid_metrics)


In [ ]:
emulators_to_analyze = {
    EMULATOR_1_LABEL: {'ds': dataset_emu_1, 'color': EMULATOR_1_COLOR}
}

if dataset_emu_2 is not None:
    emulators_to_analyze[EMULATOR_2_LABEL] = {'ds': dataset_emu_2, 'color': EMULATOR_2_COLOR}

Here we look at the isotropized 2D power spectrum using a region of the Pacific that does not include any significant land. What this means in practice, is we are looking at the energy (at least for velocity) in the different spatial scales. Possible I should plot the isotropic spectrum rather than power spectrum for temperature, but it contrains the same information. I plot this against the physical wavenumber (in km), so that larger wavenumbers correspond to larger scale features. We often expect to see a dropoff in scale at the higher frequencies as this is a common limitation of emulators.

In [ ]:
var_to_eval = 'thetao'
lev_idx = 0
time_window = 100
lon_slice = slice(180, 243)
lat_slice = slice(-40, 35)

# Metrics for calculation
dx_mean = float(dataset_truth.dx.sel({'x': lon_slice, 'y': lat_slice}).mean().values)
dy_mean = float(dataset_truth.dy.sel({'x': lon_slice, 'y': lat_slice}).mean().values)

def get_spectrum(ds_target, var_name, lev_i, t_window, x_sl, y_sl):
    """Wrapper to prepare data and run torch spectrum function."""
    data_slice = ds_target[var_name].isel(lev=lev_i, time=slice(None, t_window))
    data_slice = data_slice.transpose('time', ...).sel({'x': x_sl, 'y': y_sl}).fillna(0).values
    
    k_cent, spec = compute_isotropic_spectrum_torch(
        torch.as_tensor(data_slice),
        dx=dx_mean, dy=dy_mean, n_factor=2,
        detrend='linear', window='hann', cutoff_before_bins=False
    )
    return k_cent.cpu().numpy() * 1000 * 2 * np.pi, spec.cpu().numpy()

# Compute Spectra
k_ref, spec_ref = get_spectrum(dataset_truth, var_to_eval, lev_idx, time_window, lon_slice, lat_slice)
k_emu1, spec_emu1 = get_spectrum(dataset_emu_1, var_to_eval, lev_idx, time_window, lon_slice, lat_slice)

k_emu2, spec_emu2 = (None, None)
if dataset_emu_2 is not None:
    k_emu2, spec_emu2 = get_spectrum(dataset_emu_2, var_to_eval, lev_idx, time_window, lon_slice, lat_slice)

# Plotting
plt.figure(figsize=(6, 4))

# Plot Truth
plt.loglog(1/k_ref, spec_ref.mean(0), '-', color=TRUTH_COLOR, label=TRUTH_LABEL, linewidth=2)

# Plot Emulator 1
plt.loglog(1/k_emu1, spec_emu1.mean(0), '-', color=EMULATOR_1_COLOR, label=EMULATOR_1_LABEL)

# Plot Emulator 2 (if exists)
if dataset_emu_2 is not None:
    plt.loglog(1/k_emu2, spec_emu2.mean(0), '-', color=EMULATOR_2_COLOR, label=EMULATOR_2_LABEL)

plt.gca().invert_xaxis()
plt.xlabel(r'Wavenumber $\kappa$ (km)')
plt.ylabel('Power Spectral Density')
plt.title(f'Isotropic Power Spectrum of {var_to_eval} ({dataset_truth.lev.values[lev_idx]}m)')
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.show()

Here we look at how the ocean takes up heat over the inference rollout. To remove the shorter signal corresponding to seasonality, we remove climatology before computing this values. We look at this over the upper ocean, mid depth, and deep ocean as we expect signals of decreasing strength as we go to the deep ocean. We first compute OHC as  $$OHC(t) = \sum_{k \in Z} \sum_{j} \sum_{i} \rho_0 c_p \theta_{i,j,k,t} \Delta z_{k} A_{i,j}$$

We then compute the anomaly as $$\Delta OHC(t) = OHC(t)-OHC(0)$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy.stats import pearsonr

# Constants
RHO = 1035          # Reference density (kg/m^3)
CP = 3850           # Specific heat capacity (J/kg/K)
SCALE_FACTOR = 1e21 # Scale to ZJ (10^21 J)

DEPTH_SLICES = [
    {'min': 0, 'max': 700, 'title': 'Upper Ocean (0-700m)'},
    {'min': 700, 'max': 2000, 'title': 'Mid Ocean (700-2000m)'},
    {'min': 2000, 'max': 6500, 'title': 'Deep Ocean (2000-6500m)'},
]

def get_trend(timeseries):
    """
    Calculates the linear trend of a time series in units per decade.
    """
    # Convert time to fractional years for polyfit
    try:
        # Check if we can use dt accessor
        time_as_years = (timeseries.time.dt.year + timeseries.time.dt.dayofyear/365.25 - float(timeseries.time.dt.year[0]))
    except:
        # Fallback for simple integer time steps if datetime not available
        days = (timeseries.time - timeseries.time[0]).astype(float) / 1e9 / 86400
        time_as_years = days / 365.25
        

    # Handle NaNs if any exist in the timeseries
    valid_idx = np.isfinite(timeseries.values)
    if valid_idx.sum() > 1:
        coeffs = np.polyfit(time_as_years[valid_idx], timeseries.values[valid_idx], 1)
        slope_per_decade = coeffs[0] * 10
        
        trend_line = xr.DataArray(
            np.polyval(coeffs, time_as_years),
            dims=['time'],
            coords={'time': timeseries.time}
        )
    else:
        slope_per_decade = np.nan
        trend_line = timeseries * np.nan
    
    return slope_per_decade, trend_line

def compute_ohc_anomaly(ds, depth_slice, area):
    """
    Optimized OHC calculation:
    1. Select depth slice
    2. Integrate spatially FIRST (reduces 4D -> 1D)
    3. Compute climatology/anomaly on the 1D time series
    """
    thetao_slice = ds['thetao'].sel(lev=depth_slice)
    dz_slice = ds['dz'].sel(lev=depth_slice)
    
    # 1. Spatially Integrate Absolute OHC
    # Result is a 1D Time Series. We call compute() here to load only the 1D array into memory.
    ohc_absolute_1d = (
        RHO * CP * thetao_slice * dz_slice * area
    ).sum(['x', 'y', 'lev']).compute() / SCALE_FACTOR

    # 2. Compute Anomaly on 1D series
    # Using 'dayofyear' to handle 5-day frequency data correctly
    climatology = ohc_absolute_1d.groupby('time.dayofyear').mean('time')
    ohc_anomaly = ohc_absolute_1d.groupby('time.dayofyear') - climatology
    
    return ohc_anomaly

def plot_ohc_comparison(ds_truth, emulators_dict):
    """
    Generates a multi-panel plot comparing multiple emulators against ground truth OHC.
    """
    num_plots = len(DEPTH_SLICES)
    units = 'ZJ'
    variable = r'$\Delta$ OHC'
    area = ds_truth['areacello'] # Use area from truth dataset
    
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(
        num_plots, 1, 
        figsize=(10, 3.5 * num_plots), 
        sharex=True,
        constrained_layout=True
    )
    if num_plots == 1: axes = [axes]
    
    for i, Slices in enumerate(DEPTH_SLICES):
        ax = axes[i]
        min_d, max_d = Slices['min'], Slices['max']
        depth_slice = slice(min_d, max_d)
        
        # --- Process Truth ---
        # Calculate Anomaly (Optimized)
        ts_truth_full = compute_ohc_anomaly(ds_truth, depth_slice, area)
        
        # Zero the start
        ts_truth = ts_truth_full - ts_truth_full.isel(time=0)
        
        trend_val_truth, trend_line_truth = get_trend(ts_truth)
        
        # Plot Truth
        ts_truth.plot(ax=ax, label=TRUTH_LABEL, color=TRUTH_COLOR, lw=2.5, zorder=10)
        ax.plot(trend_line_truth.time, trend_line_truth, 
                label=f'Truth Trend ({trend_val_truth:+.2e} {units}/dec)', 
                color='gray', ls='--', lw=2, zorder=10)

        # --- Process Emulators ---
        stats_text = []
        
        for label, emu_data in emulators_dict.items():
            ds_emu = emu_data['ds']
            color = emu_data['color']
            
            # Calculate Anomaly (Optimized)
            ts_emu_full = compute_ohc_anomaly(ds_emu, depth_slice, area)
            
            # Zero the start
            ts_emu = ts_emu_full - ts_emu_full.isel(time=0)
            
            trend_val_emu, trend_line_emu = get_trend(ts_emu)
            
            # Metrics
            min_len = min(len(ts_truth), len(ts_emu))
            r_val, _ = pearsonr(ts_truth.values[:min_len], ts_emu.values[:min_len])
            r2_val = r_val**2
            stats_text.append(f"{label} R²={r2_val:.2f}")

            # Plot Emulator
            ts_emu.plot(ax=ax, label=label, color=color, lw=2)
            ax.plot(trend_line_emu.time, trend_line_emu, 
                    label=f'{label} Trend ({trend_val_emu:+.2e})', 
                    color=color, ls=':', lw=1.5)

        # --- Formatting ---
        stats_title = " | ".join(stats_text)
        ax.set_title(f"{Slices['title']}\n{stats_title}", fontsize=12, loc='left')
        ax.set_ylabel(f"Global {variable} [{units}]")
        ax.grid(True, linestyle=':', alpha=0.7)
        ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1, 1))
        ax.set_xlabel('')

    axes[-1].set_xlabel("Year", fontsize=12)
    fig.suptitle(f"Global Total Change in Ocean Heat Content Anomaly by Depth", fontsize=14, y=1.02)
    plt.show()

# Run Analysis
plot_ohc_comparison(dataset_truth, emulators_to_analyze)

In this block, we look at how the time series of the mean of variables over the same depth slices as above. We expect that most of the seasonal signal occurs in the upper 700m and below we mostly track slower signals.

In [ ]:
def plot_mean_timeseries_comparison(ds_truth, emulators_dict, variable_name, var_units, var_label):
    """
    Generates a multi-panel plot comparing emulator and ground truth for the 
    area/volume-weighted mean of a specified variable.
    """
    
    num_plots = len(DEPTH_SLICES)
    area = ds_truth['areacello']
    
    # Create a wet mask based on Truth (NaN check on first time step)
    # 1.0 where water exists, 0.0 where land/NaN
    wet_mask = xr.where(np.isnan(ds_truth[variable_name].isel(time=0)), 0.0, 1.0)
    
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(
        num_plots, 1, 
        figsize=(10, 3.5 * num_plots), 
        sharex=True,
        constrained_layout=True
    )
    if num_plots == 1: axes = [axes]
    
    for i, Slices in enumerate(DEPTH_SLICES):
        ax = axes[i]
        min_d, max_d = Slices['min'], Slices['max']
        depth_slice = slice(min_d, max_d)
        
        # --- Process Truth ---
        field_truth = ds_truth[variable_name].sel(lev=depth_slice)
        dz_truth = ds_truth['dz'].sel(lev=depth_slice)
        vol_weight_truth = dz_truth * area
        
        # 1. Calculate Numerator (Sum of Field * Volume) -> Reduces 4D to 1D
        numerator_truth = (field_truth * vol_weight_truth).sum(['x', 'y', 'lev']).compute()
        
        # 2. Calculate Denominator (Total Valid Volume)
        # Using wet_mask to ensure we sum the same volume domain
        volume_denominator_truth = (vol_weight_truth * wet_mask.sel(lev=depth_slice)).sum(['x', 'y', 'lev']).compute()
        
        # 3. Compute Mean Time Series
        ts_truth = numerator_truth / volume_denominator_truth
        
        trend_val_truth, trend_line_truth = get_trend(ts_truth)
        
        # Plot Truth
        ts_truth.plot(ax=ax, label=TRUTH_LABEL, color=TRUTH_COLOR, lw=2.5, zorder=10)
        ax.plot(trend_line_truth.time, trend_line_truth, 
                label=f'Truth Trend ({trend_val_truth:+.2e} {var_units}/dec)', 
                color='gray', ls='--', lw=2, zorder=10)
        
        # --- Process Emulators ---
        stats_text = []
        
        for label, emu_data in emulators_dict.items():
            ds_emu = emu_data['ds']
            color = emu_data['color']
            
            field_emu = ds_emu[variable_name].sel(lev=depth_slice)
            dz_emu = ds_emu['dz'].sel(lev=depth_slice)
            vol_weight_emu = dz_emu * area
            
            # 1. Calculate Numerator
            numerator_emu = (field_emu * vol_weight_emu).sum(['x', 'y', 'lev']).compute()
            
            # 2. Calculate Denominator 
            # (Using emulator dz but truth mask to compare consistent domains)
            volume_denominator_emu = (vol_weight_emu * wet_mask.sel(lev=depth_slice)).sum(['x', 'y', 'lev']).compute()
            
            # 3. Compute Mean
            ts_emu = numerator_emu / volume_denominator_emu
            
            trend_val_emu, trend_line_emu = get_trend(ts_emu)
            
            # Metrics
            min_len = min(len(ts_truth), len(ts_emu))
            r_val, _ = pearsonr(ts_truth.values[:min_len], ts_emu.values[:min_len])
            r2_val = r_val**2
            stats_text.append(f"{label} R²={r2_val:.2f}")
            
            # Plot Emulator
            ts_emu.plot(ax=ax, label=label, color=color, lw=2)
            ax.plot(trend_line_emu.time, trend_line_emu, 
                    label=f'{label} Trend ({trend_val_emu:+.2e})', 
                    color=color, ls=':', lw=1.5)
            
        # --- Formatting ---
        stats_title = " | ".join(stats_text)
        ax.set_title(f"{Slices['title']}\n{stats_title}", fontsize=12, loc='left')
        ax.set_ylabel(f"Global Mean {var_label} [{var_units}]", fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.7)
        ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1, 1))
        ax.set_xlabel('')

    axes[-1].set_xlabel("Year", fontsize=12)
    fig.suptitle(f"Global Mean {var_label} Time Series by Depth", fontsize=14, y=1.02)
    plt.show()


plot_mean_timeseries_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='thetao', 
    var_units='°C', 
    var_label='Temperature'
)

In [ ]:
plot_mean_timeseries_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='so', 
    var_units='psu', 
    var_label='Salinity'
)

In this block, we look at how the time series of the variance of variables over the same depth slices as above. This is quantified as the spatial variance at each time step. This is an additional check as the mean time series represents a heavy reduction in the information on emulator skill. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from scipy.stats import pearsonr

def plot_variance_timeseries_comparison(ds_truth, emulators_dict, variable_name, var_units, var_label):
    """
    Generates a multi-panel plot comparing emulator and ground truth for the 
    spatial variance of a specified variable.
    
    Optimization:
    Uses Var(X) = E[X^2] - (E[X])^2 to avoid broadcasting the mean time series 
    back across the 4D array.
    """
    
    num_plots = len(DEPTH_SLICES)
    area = ds_truth['areacello']
    
    # Create wet mask from Truth
    wet_mask = xr.where(np.isnan(ds_truth[variable_name].isel(time=0)), 0.0, 1.0)
    
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(
        num_plots, 1, 
        figsize=(10, 3.5 * num_plots), 
        sharex=True,
        constrained_layout=True
    )
    if num_plots == 1: axes = [axes]
    
    for i, Slices in enumerate(DEPTH_SLICES):
        ax = axes[i]
        min_d, max_d = Slices['min'], Slices['max']
        depth_slice = slice(min_d, max_d)
        
        # --- Process Truth ---
        field_truth = ds_truth[variable_name].sel(lev=depth_slice)
        dz_truth = ds_truth['dz'].sel(lev=depth_slice)
        vol_weight_truth = dz_truth * area
        
        # Calculate Denominator (Total Volume)
        volume_denom_truth = (vol_weight_truth * wet_mask.sel(lev=depth_slice)).sum(['x', 'y', 'lev']).compute()
        
        # Term 1: E[X] (Weighted Global Mean)
        numerator_mean_truth = (field_truth * vol_weight_truth).sum(['x', 'y', 'lev']).compute()
        global_mean_truth = numerator_mean_truth / volume_denom_truth
        
        # Term 2: E[X^2] (Weighted Mean of Squares)
        # We square the field element-wise FIRST, then sum. This is memory efficient.
        numerator_sq_truth = ((field_truth**2) * vol_weight_truth).sum(['x', 'y', 'lev']).compute()
        global_sq_mean_truth = numerator_sq_truth / volume_denom_truth
        
        # Variance = E[X^2] - (E[X])^2
        variance_truth = global_sq_mean_truth - (global_mean_truth**2)
        
        trend_val_truth, trend_line_truth = get_trend(variance_truth)
        
        # Plot Truth
        variance_truth.plot(ax=ax, label=TRUTH_LABEL, color=TRUTH_COLOR, lw=2.5, zorder=10)
        ax.plot(trend_line_truth.time, trend_line_truth, 
                label=f'Truth Trend ({trend_val_truth:+.2e} {var_units}$^2$/dec)', 
                color='gray', ls='--', lw=2, zorder=10)
        
        # --- Process Emulators ---
        stats_text = []
        
        for label, emu_data in emulators_dict.items():
            ds_emu = emu_data['ds']
            color = emu_data['color']
            
            field_emu = ds_emu[variable_name].sel(lev=depth_slice)
            dz_emu = ds_emu['dz'].sel(lev=depth_slice)
            vol_weight_emu = dz_emu * area
            
            # Use Truth Mask for consistency in denominator
            volume_denom_emu = (vol_weight_emu * wet_mask.sel(lev=depth_slice)).sum(['x', 'y', 'lev']).compute()
            
            # Term 1: E[X]
            numerator_mean_emu = (field_emu * vol_weight_emu).sum(['x', 'y', 'lev']).compute()
            global_mean_emu = numerator_mean_emu / volume_denom_emu
            
            # Term 2: E[X^2]
            numerator_sq_emu = ((field_emu**2) * vol_weight_emu).sum(['x', 'y', 'lev']).compute()
            global_sq_mean_emu = numerator_sq_emu / volume_denom_emu
            
            # Variance
            variance_emu = global_sq_mean_emu - (global_mean_emu**2)
            
            trend_val_emu, trend_line_emu = get_trend(variance_emu)
            
            # Metrics
            min_len = min(len(variance_truth), len(variance_emu))
            r_val, _ = pearsonr(variance_truth.values[:min_len], variance_emu.values[:min_len])
            r2_val = r_val**2
            stats_text.append(f"{label} R²={r2_val:.2f}")
            
            # Plot Emulator
            variance_emu.plot(ax=ax, label=label, color=color, lw=2)
            ax.plot(trend_line_emu.time, trend_line_emu, 
                    label=f'{label} Trend ({trend_val_emu:+.2e})', 
                    color=color, ls=':', lw=1.5)
            
        # --- Formatting ---
        stats_title = " | ".join(stats_text)
        ax.set_title(f"{Slices['title']}\n{stats_title}", fontsize=12, loc='left')
        ax.set_ylabel(f"Global Variance {var_label} [{var_units}$^2$]", fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.7)
        ax.legend(fontsize=9, loc='upper left', bbox_to_anchor=(1, 1))
        ax.set_xlabel('')

    axes[-1].set_xlabel("Year", fontsize=12)
    fig.suptitle(f"Global Spatial Variance of {var_label} Time Series by Depth", fontsize=14, y=1.02)
    plt.show()

# --- Execution ---

# 1. Plot Temperature Variance
plot_variance_timeseries_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='thetao', 
    var_units='°C', 
    var_label='Temperature'
)

Here we look at the mean bias in each major ocean basin. This bias is computed as the difference between the mean state of the emulator and the mean state of OM4, as such this can average out a lot of errors.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import xarray as xr
import cmocean

# Configuration
BASINS_TO_PLOT = ['Atlantic', 'Pacific', 'Indian', 'Southern']


def calculate_zonal_mean_simple(field_mean, mask, dx, wet):
    """
    Simple Zonal Mean using your exact method.
    Numerator: Sum(Field * Mask * dx)
    Denominator: Sum(Mask * Wet_Truth * dx)
    """
    numerator = (field_mean * mask * dx).sum('x')
    denominator = (mask * wet * dx).sum('x')
    return numerator / denominator

def plot_zonal_bias_comparison(ds_truth, emulators_dict, variable_name, units, masks):
    """Generates a 2x2 plot of zonal mean bias for EACH emulator provided."""
    
    # 0. Setup Truth Metrics (Used for ALL calculations)
    truth_field_mean = ds_truth[variable_name].mean('time')
    dx_truth = ds_truth['dx']
    
    # Define "Wet" mask from Truth (Matches your snippet: 1 where Data, NaN where Land)
    wet_mask = (ds_truth[variable_name].isel(time=0) * 0 + 1).compute()
    
    # 1. Calculate Truth Profiles
    truth_profiles = {}
    print("Calculating Truth Zonal Profiles...")
    for basin in BASINS_TO_PLOT:
        truth_profiles[basin] = calculate_zonal_mean_simple(
            truth_field_mean, masks[basin], dx_truth, wet_mask
        ).compute()

    # 2. Iterate over Emulators
    for label, emu_data in emulators_dict.items():
        ds_emu = emu_data['ds']
        emu_field_mean = ds_emu[variable_name].mean('time')
        
        print(f"\nGenerating Zonal Bias Plot for {label}...")
        
        bias_profiles = {}
        rmse_vals = {}
        
        # Weights for RMSE (Mean across X)
        weight_2d = (ds_truth['dy'] * ds_truth['dz']).mean('x').compute()

        all_biases = []

        for basin in BASINS_TO_PLOT:
            # Calculate Emulator Profile using TRUTH metrics
            prof_emu = calculate_zonal_mean_simple(
                emu_field_mean, masks[basin], dx_truth, wet_mask
            ).compute()
            
            prof_truth = truth_profiles[basin]
            
            # Simple Subtraction
            bias = prof_emu - prof_truth
            bias_profiles[basin] = bias.drop('time')
            all_biases.append(bias.values.flatten())
            
            # RMSE Calculation
            sq_diff = bias**2
            rmse = np.sqrt((sq_diff * weight_2d).sum(['y','lev']) / weight_2d.sum(['y','lev']))
            rmse_vals[basin] = rmse.item()

        # Dynamic Color Scaling
        flat_biases = np.concatenate(all_biases)
        flat_biases = flat_biases[~np.isnan(flat_biases)]
        max_abs_bias = np.percentile(np.abs(flat_biases), 98) if len(flat_biases) > 0 else 1.0
        
        norm = colors.Normalize(vmin=-max_abs_bias, vmax=max_abs_bias)
        cmap = cmocean.cm.balance

        # --- Plotting ---
        plt.style.use('seaborn-v0_8-talk')
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True, constrained_layout=True)
        axes = axes.flatten()

        for i, basin in enumerate(BASINS_TO_PLOT):
            ax = axes[i]
            p_mesh = bias_profiles[basin].plot.contourf(
                ax=ax, cmap=cmap, norm=norm, add_colorbar=False, 
                levels=np.linspace(-max_abs_bias, max_abs_bias, 25), yincrease=False,
                x='y', y='lev' 
            )
            ax.set_title(f"{basin} (RMSE: {rmse_vals[basin]:.3f} {units})", fontsize=16, loc='left')
            ax.set_facecolor('grey')
            ax.set_xlabel(''); ax.set_ylabel('')

        axes[0].set_ylabel('Depth (m)'); axes[2].set_ylabel('Depth (m)')
        axes[2].set_xlabel('Latitude index'); axes[3].set_xlabel('Latitude index')
        cbar = fig.colorbar(p_mesh, ax=axes, shrink=0.8, pad=0.02)
        cbar.set_label(f"Bias ({units})")
        fig.suptitle(f"Zonal Mean Bias: {label} vs Truth ({variable_name})", fontsize=20, y=1.03)
        plt.show()
        
# --- Execution ---
basin_masks = extract_basin_masks(dataset_truth)

plot_zonal_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='thetao', 
    units='$^\circ$C', 
    masks=basin_masks
)

In [ ]:
# 3. Run Analysis for Salinity (Optional)
plot_zonal_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='so', 
    units='PSU', 
    masks=basin_masks
)

This is the same bias as above, but seperated into depth levels rather than basins.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import xarray as xr
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Configuration
VARIABLE_TO_PLOT = 'thetao' # Options: 'thetao', 'so'

DEPTH_SLICES = [
    {'min': 0, 'max': 700, 'title': 'Upper Ocean (0-700m)'},
    {'min': 700, 'max': 2000, 'title': 'Mid-Ocean (700-2000m)'},
    {'min': 2000, 'max': 6000, 'title': 'Deep Ocean (2000m+)'},
]

def calculate_vertical_mean_simple(field, dz, wet_mask, min_depth, max_depth):
    """
    Calculates the vertical mean map using Truth's geometry.
    Formula: Sum(Field * dz * wet) / Sum(dz * wet) over depth slice.
    """
    # Select depth slices
    f_slice = field.sel(lev=slice(min_depth, max_depth))
    dz_slice = dz.sel(lev=slice(min_depth, max_depth))
    wet_slice = wet_mask.sel(lev=slice(min_depth, max_depth))
    
    numerator = (f_slice * dz_slice * wet_slice).sum('lev')
    denominator = (dz_slice * wet_slice).sum('lev')
    
    return numerator / denominator

def rmse_spatial(truth_map, pred_map, area):
    """Area-weighted RMSE for 2D spatial maps."""
    sq_diff = (truth_map - pred_map)**2
    
    # Mask weights where data is missing
    weights = area.where(sq_diff.notnull())
    
    numerator = (sq_diff * weights).sum(['y', 'x'])
    denominator = weights.sum(['y', 'x'])
    
    return np.sqrt(numerator / denominator).values.item()

def plot_spatial_bias_comparison(ds_truth, emulators_dict, variable_name, units):
    """
    Generates spatial bias maps for each emulator and depth slice.
    """
    
    # 0. Setup Truth Metrics (Used for ALL calculations)
    print("Calculating Truth Time Mean and Masks...")
    truth_field_mean = ds_truth[variable_name].mean('time')
    dz_truth = ds_truth['dz']
    area = ds_truth['areacello']
    
    # Define "Wet" mask from Truth (1 where Data, NaN where Land)
    wet_mask = (ds_truth[variable_name].isel(time=0) * 0 + 1).compute()

    # 1. Calculate Truth Maps for all slices
    truth_maps = {}
    print("Calculating Truth Spatial Maps...")
    for Slices in DEPTH_SLICES:
        key = Slices['title']
        truth_maps[key] = calculate_vertical_mean_simple(
            truth_field_mean, dz_truth, wet_mask, Slices['min'], Slices['max']
        ).compute()

    # 2. Iterate over Emulators
    for label, emu_data in emulators_dict.items():
        ds_emu = emu_data['ds']
        emu_field_mean = ds_emu[variable_name].mean('time')
        
        print(f"\nGenerating Spatial Bias Maps for {label}...")
        
        bias_maps = {}
        rmse_vals = {}
        all_biases_flat = []
        
        for Slices in DEPTH_SLICES:
            key = Slices['title']
            
            # Compute Emulator Map using TRUTH metrics (dz, wet_mask)
            map_emu = calculate_vertical_mean_simple(
                emu_field_mean, dz_truth, wet_mask, Slices['min'], Slices['max']
            ).compute()
            
            map_truth = truth_maps[key]
            
            # Bias
            bias = map_emu - map_truth
            bias_maps[key] = bias
            all_biases_flat.append(bias.values.flatten())
            
            # RMSE
            rmse_vals[key] = rmse_spatial(map_truth, map_emu, area)

        # Dynamic Color Scaling
        flat_data = np.concatenate(all_biases_flat)
        flat_data = flat_data[~np.isnan(flat_data)]
        
        if len(flat_data) > 0:
            max_abs_bias = np.percentile(np.abs(flat_data), 98)
            print(f"  > Bias Range (98th percentile): +/- {max_abs_bias:.3f} {units}")
        else:
            max_abs_bias = 1.0

        vmin, vmax = -max_abs_bias, max_abs_bias
        norm = colors.Normalize(vmin=vmin, vmax=vmax)
        cmap = cmocean.cm.balance

        # --- Plotting ---
        num_slices = len(DEPTH_SLICES)
        fig, axes = plt.subplots(
            1, num_slices, 
            figsize=(6 * num_slices, 5),
            subplot_kw={'projection': ccrs.Robinson(central_longitude=210)},
            constrained_layout=True
        )
        if num_slices == 1: axes = [axes]
        
        for i, Slices in enumerate(DEPTH_SLICES):
            key = Slices['title']
            ax = axes[i]
            bias = bias_maps[key]
            
            # pcolormesh needs coordinates. 
            p = bias.plot.pcolormesh(
                ax=ax, transform=ccrs.PlateCarree(),
                x='x', y='y',
                cmap=cmap, norm=norm, add_colorbar=False,
                rasterized=True
            )
            
            ax.set_title(f"{key}\nRMSE: {rmse_vals[key]:.3f} {units}", fontsize=14)
            ax.add_feature(cfeature.LAND, zorder=10, edgecolor='black', facecolor='lightgray')
            ax.coastlines(zorder=11)
            ax.gridlines(draw_labels=False, alpha=0.3)

        # Shared Colorbar
        cbar = fig.colorbar(p, ax=axes, orientation='horizontal', shrink=0.4, pad=0.05, aspect=30)
        cbar.set_label(f"Bias ({units})", fontsize=12)
        
        fig.suptitle(f"Time-Mean Bias: {label} vs Truth ({variable_name})", fontsize=18, y=1.1)
        plt.show()

# --- Execution ---
if VARIABLE_TO_PLOT == 'thetao':
    var_label, units = 'Temperature', '$^\circ$C'
else:
    var_label, units = 'Salinity', 'PSU'

plot_spatial_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name=VARIABLE_TO_PLOT, 
    units=units
)

In [ ]:
# Configuration
VARIABLE_TO_PLOT = 'so' # Options: 'thetao', 'so'

# --- Execution ---
if VARIABLE_TO_PLOT == 'thetao':
    var_label, units = 'Temperature', '$^\circ$C'
else:
    var_label, units = 'Salinity', 'PSU'

plot_spatial_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name=VARIABLE_TO_PLOT, 
    units=units
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import xarray as xr
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Configuration
VARIABLE_TO_PLOT = 'thetao' # Options: 'thetao', 'so'

DEPTH_SLICES = [
    {'min': 0, 'max': 700, 'title': 'Upper Ocean (0-700m)'},
    {'min': 700, 'max': 2000, 'title': 'Mid-Ocean (700-2000m)'},
    {'min': 2000, 'max': 6000, 'title': 'Deep Ocean (2000m+)'},
]

def calculate_depth_averaged_variance(dataset, var_name, dz, wet_mask, min_depth, max_depth):
    """
    Calculates the vertically averaged TEMPORAL variance map.
    """
    # Select depth slices
    field_slice = dataset[var_name].sel(lev=slice(min_depth, max_depth))
    dz_slice = dz.sel(lev=slice(min_depth, max_depth))
    wet_slice = wet_mask.sel(lev=slice(min_depth, max_depth))
    
    # 1. Calculate Temporal Variance (3D: lev, y, x)
    variance_3d = field_slice.var('time', skipna=True)
    
    # 2. Vertical Average (Weighted by dz)
    numerator = (variance_3d * dz_slice * wet_slice).sum('lev')
    denominator = (dz_slice * wet_slice).sum('lev')
    
    return numerator / denominator

def rmse_spatial(truth_map, pred_map, area):
    """Area-weighted RMSE for 2D spatial maps."""
    sq_diff = (truth_map - pred_map)**2
    weights = area.where(sq_diff.notnull())
    numerator = (sq_diff * weights).sum(['y', 'x'])
    denominator = weights.sum(['y', 'x'])
    return np.sqrt(numerator / denominator).values.item()

def plot_variance_bias_comparison(ds_truth, emulators_dict, variable_name, units):
    """
    Generates spatial bias maps of TEMPORAL VARIANCE for each emulator.
    Configured to use a separate colorbar for each panel, CONSISTENT across emulators
    (locked to the range of the first emulator processed).
    """
    
    # 0. Setup Truth Metrics
    print("Calculating Truth Metrics...")
    dz_truth = ds_truth['dz']
    area = ds_truth['areacello']
    
    # Define "Wet" mask from Truth
    wet_mask = (ds_truth[variable_name].isel(time=0) * 0 + 1).compute()

    # 1. Calculate Truth Variance Maps
    truth_maps = {}
    print("Calculating Truth Variance Maps...")
    for Slices in DEPTH_SLICES:
        key = Slices['title']
        truth_maps[key] = calculate_depth_averaged_variance(
            ds_truth, variable_name, dz_truth, wet_mask, Slices['min'], Slices['max']
        ).compute()

    # Dictionary to store color limits from the first emulator
    shared_limits = {}

    # 2. Iterate over Emulators
    for idx, (label, emu_data) in enumerate(emulators_dict.items()):
        ds_emu = emu_data['ds']
        
        print(f"\nGenerating Variance Bias Maps for {label}...")
        
        bias_maps = {}
        rmse_vals = {}
        
        for Slices in DEPTH_SLICES:
            key = Slices['title']
            
            # Compute Emulator Variance Map using TRUTH metrics
            map_emu = calculate_depth_averaged_variance(
                ds_emu, variable_name, dz_truth, wet_mask, Slices['min'], Slices['max']
            ).compute()
            
            map_truth = truth_maps[key]
            
            # Bias (Diff in Variance)
            bias_maps[key] = map_emu - map_truth
            
            # RMSE
            rmse_vals[key] = rmse_spatial(map_truth, map_emu, area)

        # --- Plotting ---
        num_slices = len(DEPTH_SLICES)
        fig, axes = plt.subplots(
            1, num_slices, 
            figsize=(6 * num_slices, 5),
            subplot_kw={'projection': ccrs.Robinson(central_longitude=210)},
            constrained_layout=True
        )
        if num_slices == 1: axes = [axes]
        
        cmap = cmocean.cm.balance
        
        for i, Slices in enumerate(DEPTH_SLICES):
            key = Slices['title']
            ax = axes[i]
            bias = bias_maps[key]
            
            # Determine Color Scaling (Lock to First Emulator)
            if idx == 0:
                flat_bias = bias.values.flatten()
                flat_bias = flat_bias[~np.isnan(flat_bias)]
                
                if len(flat_bias) > 0:
                    local_max = np.percentile(np.abs(flat_bias), 98)
                else:
                    local_max = 1.0
                
                shared_limits[key] = local_max
                print(f"  > {key} Scale locked to: +/- {local_max:.3e} {units}^2")
            
            limit = shared_limits[key]
            norm = colors.Normalize(vmin=-limit, vmax=limit)
            
            # Plot with INDIVIDUAL colorbar
            p = bias.plot.pcolormesh(
                ax=ax, transform=ccrs.PlateCarree(),
                x='x', y='y',
                cmap=cmap, norm=norm, 
                add_colorbar=True,
                cbar_kwargs={
                    'orientation': 'horizontal', 
                    'shrink': 0.8, 
                    'pad': 0.08, 
                    'label': f"Bias ({units}" + r"$^2$)"
                },
                rasterized=True
            )
            
            ax.set_title(f"{key}\nRMSE: {rmse_vals[key]:.2e} {units}" + r"$^2$", fontsize=14)
            ax.add_feature(cfeature.LAND, zorder=10, edgecolor='black', facecolor='lightgray')
            ax.coastlines(zorder=11)
            ax.gridlines(draw_labels=False, alpha=0.3)
        
        fig.suptitle(f"Difference in Temporal Variance: {label} vs Truth ({variable_name})", fontsize=18, y=1.05)
        plt.show()

# --- Execution ---
if VARIABLE_TO_PLOT == 'thetao':
    var_label, units = 'Temperature', '$^\circ$C'
else:
    var_label, units = 'Salinity', 'PSU'

plot_variance_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name=VARIABLE_TO_PLOT, 
    units=units
)

In [ ]:
plot_variance_bias_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name='vo', 
    units='m/s'
)

In [ ]:
# --------------------------------------------------------------------------------
# BLOCK 11: Zonal Variance Bias Analysis (Lat-Depth Cross Sections)
# --------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import xarray as xr
import cmocean

# Configuration
BASINS_TO_PLOT = ['Atlantic', 'Pacific', 'Indian', 'Southern']

def extract_basin_masks(ds_with_basin_var):
    """Extracts boolean masks from the integer 'basin' variable."""
    basin_code = ds_with_basin_var['basin']
    return {
        'Atlantic': xr.where(basin_code == 2, 1.0, np.nan),
        'Pacific': xr.where(basin_code == 3, 1.0, np.nan),
        'Indian': xr.where(basin_code == 5, 1.0, np.nan),
        'Southern': xr.where(basin_code == 1, 1.0, np.nan),
    }

def calculate_zonal_mean_simple(field_2d, mask, dx, wet):
    """
    Simple Zonal Mean using Truth's geometry.
    Numerator: Sum(Field * Mask * dx)
    Denominator: Sum(Mask * Wet_Truth * dx)
    """
    numerator = (field_2d * mask * dx).sum('x')
    denominator = (mask * wet * dx).sum('x')
    return numerator / denominator

def plot_zonal_variance_comparison(ds_truth, emulators_dict, variable_name, units, masks):
    """
    Generates a 2x2 plot of Zonal Variance Bias (Emulator_Var - Truth_Var)
    for EACH emulator provided.
    """
    
    # 0. Setup Truth Metrics
    print("Calculating Truth Temporal Variance...")
    # Calculate Temporal Variance at each grid cell (collapses Time dim)
    truth_field_var = ds_truth[variable_name].var('time', skipna=True)
    dx_truth = ds_truth['dx']
    
    # Define "Wet" mask from Truth
    wet_mask = (ds_truth[variable_name].isel(time=0) * 0 + 1).compute()
    
    # 1. Calculate Truth Zonal Variance Profiles
    truth_profiles = {}
    print("Calculating Truth Zonal Variance Profiles...")
    for basin in BASINS_TO_PLOT:
        truth_profiles[basin] = calculate_zonal_mean_simple(
            truth_field_var, masks[basin], dx_truth, wet_mask
        ).compute()

    # 2. Iterate over Emulators
    for label, emu_data in emulators_dict.items():
        ds_emu = emu_data['ds']
        print(f"\nCalculating Emulator Variance for {label}...")
        
        # Calculate Emulator Temporal Variance
        emu_field_var = ds_emu[variable_name].var('time', skipna=True)
        
        bias_profiles = {}
        rmse_vals = {}
        
        # Weights for RMSE (Mean across X)
        weight_2d = (ds_truth['dy'] * ds_truth['dz']).mean('x').compute()

        all_biases = []

        for basin in BASINS_TO_PLOT:
            # Calculate Emulator Zonal Variance Profile
            prof_emu = calculate_zonal_mean_simple(
                emu_field_var, masks[basin], dx_truth, wet_mask
            ).compute()
            
            prof_truth = truth_profiles[basin]
            
            # Bias = Emu_Var - Truth_Var
            bias = prof_emu - prof_truth
            bias_profiles[basin] = bias.drop('time')
            all_biases.append(bias.values.flatten())
            
            # RMSE Calculation
            sq_diff = bias**2
            rmse = np.sqrt((sq_diff * weight_2d).sum(['y','lev']) / weight_2d.sum(['y','lev']))
            rmse_vals[basin] = rmse.item()

        # Dynamic Color Scaling
        flat_biases = np.concatenate(all_biases)
        flat_biases = flat_biases[~np.isnan(flat_biases)]
        
        if len(flat_biases) > 0:
            max_abs_bias = np.percentile(np.abs(flat_biases), 98)
            print(f"  > Variance Bias Range (98th percentile): +/- {max_abs_bias:.3e} {units}^2")
        else:
            max_abs_bias = 1.0
        
        norm = colors.Normalize(vmin=-max_abs_bias, vmax=max_abs_bias)
        cmap = cmocean.cm.balance

        # --- Plotting ---
        plt.style.use('seaborn-v0_8-talk')
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True, constrained_layout=True)
        axes = axes.flatten()

        for i, basin in enumerate(BASINS_TO_PLOT):
            ax = axes[i]
            p_mesh = bias_profiles[basin].plot.contourf(
                ax=ax, cmap=cmap, norm=norm, add_colorbar=False, 
                levels=np.linspace(-max_abs_bias, max_abs_bias, 25), yincrease=False,
                x='y', y='lev' 
            )
            
            # Scientific notation for Variance RMSE
            ax.set_title(f"{basin} (RMSE: {rmse_vals[basin]:.2e} {units}" + r"$^2$)", fontsize=16, loc='left')
            ax.set_facecolor('grey')
            ax.set_xlabel(''); ax.set_ylabel('')

        axes[0].set_ylabel('Depth (m)'); axes[2].set_ylabel('Depth (m)')
        axes[2].set_xlabel('Latitude index'); axes[3].set_xlabel('Latitude index')
        
        cbar = fig.colorbar(p_mesh, ax=axes, shrink=0.8, pad=0.02)
        cbar.set_label(f"Variance Bias ({units}" + r"$^2$)")
        
        fig.suptitle(f"Zonal Variance Bias: {label} vs Truth ({variable_name})", fontsize=20, y=1.03)
        plt.show()

# --- Execution ---
basin_masks = extract_basin_masks(dataset_truth)

if VARIABLE_TO_PLOT == 'thetao':
    var_label, units = 'Temperature', '$^\circ$C'
else:
    var_label, units = 'Salinity', 'PSU'

plot_zonal_variance_comparison(
    dataset_truth, 
    emulators_to_analyze, 
    variable_name=VARIABLE_TO_PLOT, 
    units=units,
    masks=basin_masks
)